# **1. 단순 선형 회귀 분석**
- 전복의 나이를 예측하는 선형회귀모델을 생성하세요.
- 전복의 ‘성별’, ‘키’, ‘지름’, ‘높이’, ‘전체무게’, ‘몸통무게’, ‘내장무게’, ‘껍질무게’를 이용해 ‘껍질의 고리 수’를 예측한 뒤, **예측된 ‘껍질의 고리 수’에 1.5를 더하면 전복의 나이**가 됩니다.

In [1]:
# 기본 모듈 불러오기
import numpy as np
import pandas as pd

**1) 데이터 load 및 변형**

In [12]:
# 데이터 로드
data = pd.read_csv("abalone.csv")
data.head()
print(data.shape)

# 성별 M은 Male, F는 Female, I는 Infant 이므로 따로 열 만들기
for label in "MFI":
    data[label] = data["Sex"] == label
data.drop('Sex', axis=1, inplace=True)

,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings,M,F,I
0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,True,False,False
1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,True,False,False
2,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,False,True,False
3,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,True,False,False
4,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,False,False,True


In [7]:
import urllib.request
import pandas as pd

# 1. UCI 저장소에서 abalone 데이터 다운로드
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"
urllib.request.urlretrieve(url, "abalone.csv")

# 2. 열 이름(Column names) 지정하여 데이터 로드
# 원본 데이터에는 헤더가 없으므로 아래와 같이 컬럼명을 지정해 줍니다.
columns = ["Sex", "Length", "Diameter", "Height", "Whole weight",
           "Shucked weight", "Viscera weight", "Shell weight", "Rings"]

data = pd.read_csv("abalone.csv", names=columns)

# 성별 원-핫 인코딩 처리
for label in "MFI":
    data[label] = data["Sex"] == label
data.drop('Sex', axis=1, inplace=True)

# 데이터 확인
display(data.head())
print("데이터 크기:", data.shape)

,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings,M,F,I
0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,True,False,False
1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,True,False,False
2,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,False,True,False
3,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,True,False,False
4,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,False,False,True


데이터 크기: (4177, 11)


**2) X, y 선택**
: y는 Rings열, X는 Rings열을 제외한 나머지를 선택하되 전부 실수가 되도록 한다.

In [13]:
# X,y 데이터 선택
y = data['Rings']

X = data.drop('Rings', axis=1).astype(float)

# 변환 확인
print("X의 데이터 타입:\n", X.dtypes)
print("X shape:", X.shape, "y shape:", y.shape)

X의 데이터 타입:
 Length            float64
Diameter          float64
Height            float64
Whole weight      float64
Shucked weight    float64
Viscera weight    float64
Shell weight      float64
M                 float64
F                 float64
I                 float64
dtype: object
X shape: (4177, 10) y shape: (4177,)


 **3) train/test set 분리**

In [14]:
# 필요한 모듈 불러오기
from sklearn.model_selection import train_test_split

In [15]:
# train과 test set 분리 (train:test = 7:3 비율로)
X = data.drop('Rings', axis=1)  # 타겟 열을 제외한 나머지 특성
y = data['Rings']               # 예측하려는 타겟 열

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 분리 결과 확인
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (2923, 10)
X_test shape: (1254, 10)


**4) 선형회귀모델 생성, 모델 예측치 구하기**

In [16]:
#필요한 모듈 불러오기
from sklearn.linear_model import LinearRegression

In [17]:
#선형회귀모델 생성 및 훈련
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [18]:
# 모델 예측치 구하기
y_pred = model.predict(X_test)
# 모델 예측치를 활용해 최종적으로 전복의 나이를 예측
predicted_age = y_pred + 1.5

# 예측 결과 일부 확인
print("예측된 고리 수(Rings):", y_pred[:5])
print("예측된 전복의 나이(Age):", predicted_age[:5])

예측된 고리 수(Rings): [11.80063634 10.25335812 14.10360339 12.04226171 11.19633484]
예측된 전복의 나이(Age): [13.30063634 11.75335812 15.60360339 13.54226171 12.69633484]


**5) 모델 평가: MSE, RMSE, R2 score, corr 구하기**

In [19]:
#필요한 모듈 불러오기
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

- MSE, RMSE

In [20]:
#mse, rmse
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("MSE:", mse)
print("RMSE:", rmse)

MSE: 4.784788642609096
RMSE: 2.187415973839703


- R2 score

In [21]:
#R2 score 측정
r2 = r2_score(y_test, y_pred)

print("R2 score:", r2)

R2 score: 0.5288038218248535


- 회귀 절편값

In [22]:
#회귀 절편 값
print("회귀 절편(Intercept):", model.intercept_)

회귀 절편(Intercept): 3.505437824183651


- 회귀 계수 값

In [23]:
#회귀 계수 값
print("회귀 계수(Coefficients):", model.coef_)

회귀 계수(Coefficients): [  0.68477866  10.22284979  10.12970323   9.96936997 -21.51854239
 -10.69779966   7.95540502   0.3067448    0.21002805  -0.51677285]


- 상관계수

Hint: corr 함수 이용.

In [24]:
# 상관계수 구하기
result_df = pd.DataFrame({'actual': y_test.values, 'pred': y_pred})
print(result_df.corr())

          actual      pred
actual  1.000000  0.728435
pred    0.728435  1.000000


# **2. Polynomial features**

In [25]:
# PolynomialFeatures 라이브러리 호출
from sklearn.preprocessing import PolynomialFeatures

In [26]:
# 임의 데이터 생성

X = np.arange(6).reshape(3, 2)

df =  pd.DataFrame(X)
df.columns = ['x_1','x_2']
df

,x_1,x_2
0,0,1
1,2,3
2,4,5


In [27]:
# 차원은 2로 설정
poly = PolynomialFeatures(degree=2)
# fit_transform 메소드를 통해 데이터 변환
poly_ftr = poly.fit_transform(df)
# PolynomialFeatures로 변환 된 데이터를 데이터 프레임 형태로 변환
poly_df = pd.DataFrame(poly_ftr, columns=poly.get_feature_names_out())
poly_df

,1,x_1,x_2,x_1^2,x_1 x_2,x_2^2
0,1.0,0.0,1.0,0.0,0.0,1.0
1,1.0,2.0,3.0,4.0,6.0,9.0
2,1.0,4.0,5.0,16.0,20.0,25.0


In [29]:
# df_poly의 컬럼을 1,x1,x2,x1^2,x1*x2,x2^2 로 변경
df_poly.columns = ['1', 'x1', 'x2', 'x1^2', 'x1*x2', 'x2^2']
df_poly

,1,x1,x2,x1^2,x1*x2,x2^2
0,1.0,0.0,1.0,0.0,0.0,1.0
1,1.0,2.0,3.0,4.0,6.0,9.0
2,1.0,4.0,5.0,16.0,20.0,25.0
